In [ ]:
import tensorflow as tf
import gymnasium as gym
from gymnasium.wrappers.atari_preprocessing import AtariPreprocessing
from gymnasium.wrappers.frame_stack import FrameStack
import numpy as np
from tqdm import tqdm
from collections import deque
import random

# --- Environment Setup 
env = gym.make('SpaceInvadersNoFrameskip-v4', render_mode='rgb_array')
env = AtariPreprocessing(env, grayscale_obs=True, frame_skip=5, terminal_on_life_loss=True, scale_obs=True)
env = FrameStack(env, num_stack=4)

# --- Hyperparameters ---
gamma = 0.99  # Discount factor
epsilon = 1.0  # Initial exploration rate
epsilon_min = 0.1  # Minimum exploration rate
epsilon_decay = 0.995  # Decay rate for exploration
batch_size = 32  # Batch size for training
memory_size = 100000  # Maximum size of replay memory
target_update_frequency = 1000  # How often to update the target network (in steps)
learning_rate = 0.00025  # Learning rate for the optimizer
num_episodes = 10000  # Increased number of episodes
max_steps_per_episode = 10000 # Maximum number of steps per episode

# --- Replay Memory ---
class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

    def __len__(self):
        return len(self.memory)

memory = ReplayMemory(memory_size)

# --- DQN Model  ---
num_actions = env.action_space.n
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (8, 8), strides=(4, 4), activation='relu', input_shape=(84, 84, 4)),
    tf.keras.layers.Conv2D(64, (4, 4), strides=(2, 2), activation='relu'),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(num_actions)
])

# --- Target Network ---
target_model = tf.keras.models.clone_model(model)
target_model.set_weights(model.get_weights())

# --- Optimizer and Loss Function ---
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
loss_function = tf.keras.losses.Huber()

# --- Update Target Network Function ---
def update_target_network(model, target_model):
    target_model.set_weights(model.get_weights())

# --- Epsilon-Greedy Action Selection ---
def choose_action(state, model, epsilon):
    if np.random.rand() <= epsilon:
        return env.action_space.sample()  # Explore
    else:
        q_values = model.predict(state, verbose=0) # Exploit
        return np.argmax(q_values[0])

# --- Training Loop ---
total_steps = 0
for episode in tqdm(range(num_episodes), desc="Training Progress"):
    state, _ = env.reset()
    # Transpose the state here after getting it from the environment
    state = np.transpose(state, (1, 2, 0))
    episode_reward = 0

    for step in range(max_steps_per_episode):
        # Choose action 
        # No need to transpose here anymore
        action = choose_action(state[np.newaxis, :], model, epsilon)

        # Take action and observe the next state, reward, and done flag
        next_state, reward, done, _, _ = env.step(action)

        # Transpose the next_state here
        next_state = np.transpose(next_state, (1, 2, 0))

        # Store experience in replay memory
        memory.add(state, action, reward, next_state, done)

        # Update current state
        state = next_state
        episode_reward += reward
        total_steps += 1

        # Update target network periodically
        if total_steps % target_update_frequency == 0:
            update_target_network(model, target_model)

        # Sample and train
        if len(memory) >= batch_size:
            states, actions, rewards, next_states, dones = memory.sample(batch_size)

            # Calculate target Q-values
            target_q_values = target_model.predict(next_states, verbose=0)
            max_target_q_values = np.max(target_q_values, axis=1)
            targets = rewards + gamma * max_target_q_values * (1 - dones)

            # Calculate Q-values and loss
            with tf.GradientTape() as tape:
                q_values = model(states)
                action_masks = tf.one_hot(actions, num_actions)
                predicted_q_values = tf.reduce_sum(q_values * action_masks, axis=1)
                loss = loss_function(targets, predicted_q_values)

            # Backpropagation and optimization
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if done:
            break

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Print episode information
    print(f"Episode: {episode + 1}, Reward: {episode_reward}, Epsilon: {epsilon:.4f}")
   

# --- Save the trained model ---
model.save('trained_space_invaders_dqn_model.keras')


In [ ]:
# Validate the trained model

# --- Load the trained model (from Step 4) ---
model = tf.keras.models.load_model('trained_space_invaders_dqn_model.keras')

# --- Hyperparameters ---
num_episodes = 10  # Number of episodes to validate
max_steps_per_episode = 10000  # Maximum number of steps per episode

# --- Validation Loop ---
total_rewards = []
for episode in tqdm(range(num_episodes), desc="Validation Progress"):
    state, _ = env.reset()
    # Transpose the state so it's compatible with model.predict
    state = np.transpose(state, (1, 2, 0))
    episode_reward = 0

    for step in range(max_steps_per_episode):
        # Choose action
        action = np.argmax(model.predict(state[np.newaxis, :], verbose=0)[0])

        # Take action and observe the next state, reward, and done flag
        next_state, reward, done, _, _ = env.step(action)

        # Transpose next_state as well
        next_state = np.transpose(next_state, (1, 2, 0))

        # Update current state and episode reward
        state = next_state
        episode_reward += reward

        if done:
            break

    total_rewards.append(episode_reward)

# Print the average reward
print(f"Average Reward: {np.mean(total_rewards)}")

Validation Progress: 100%|██████████| 10/10 [01:09<00:00,  6.95s/it]

Average Reward: 171.0
